# Notebook 11 — Held-out negation split, manual-check sheets, and final tables (CPU only, ~5 min)

1. Splits the LLM-written (held-out) negations into **adds a negation word** (e.g. "will *not* have") vs **rewrite without a negation word** (e.g. "Low" → "High", "faster" → "slower"). The second group shares nothing with the training rule, so it is the cleanest test of generalisation.
2. Exports **manual-check sheets** (50 LLM negations, 20 rule negations, 20 paraphrases per dataset) to fill in by hand; re-running the notebook afterwards reports the validity rates.
3. Writes all **final manuscript tables** to `results/final_tables/` (CSV, plus one Markdown file with every table).

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob
ROOT = '/content/drive/MyDrive/asag_revision'
sys.path.insert(0, ROOT)
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from asag_utils import *
from llm_utils import SCORE_TARGETS, LABEL_ORDER, score_from_logprobs, labels_from_logprobs
from perturb import NEG_WORDS, valid_pairs, negation_form
RES = f'{ROOT}/results'; OUT = f'{RES}/final_tables'; os.makedirs(OUT, exist_ok=True)
mo = pd.read_csv(f'{ROOT}/data/mohler_prepared.csv'); se = pd.read_csv(f'{ROOT}/data/scientsbank_prepared.csv')
LPM = [f'lp_{s}' for s in SCORE_TARGETS]; LPS = [f'lp_{l}' for l in LABEL_ORDER]
def sc(ds, s): return f'{RES}/robustness2_scores_{ds}_{s}.csv'

def boot(values, groups, n_boot=2000, seed=SEED):
    values, groups = np.asarray(values, float), np.asarray(groups)
    s_ = [np.flatnonzero(groups == g) for g in pd.unique(groups)]; rng = np.random.default_rng(seed)
    b = np.array([values[np.concatenate([s_[i] for i in rng.integers(0, len(s_), len(s_))])].mean() for _ in range(n_boot)])
    return np.percentile(b, 2.5), np.percentile(b, 97.5), min(1.0, max(2 * min((b <= 0).mean(), (b >= 0).mean()), 1 / n_boot))

Mounted at /content/drive


## 1. Held-out negations: adds a negation word vs. rewrite without one

In [2]:
def neg_count(t): return len(NEG_WORDS.findall(str(t)))
sets = {ds: pd.read_csv(f'{RES}/robustness2_set_{ds}.csv') for ds in ['mohler', 'scientsbank']}
for ds, rs in sets.items():
    orig = rs[rs.variant == 'original'].set_index('orig_uid').student_answer
    o = rs.orig_uid.map(orig)
    rs['neg_type'] = np.where([neg_count(v) > neg_count(w) for v, w in zip(rs.student_answer, o)],
                              'adds negation word', 'rewrite without negation word')
    rs['keep'] = (valid_pairs(rs) & ~((rs.variant == 'negated_llm') & (rs.student_answer.astype(str) == o.astype(str)))).values
    print(ds, rs[(rs.variant == 'negated_llm') & rs.keep].neg_type.value_counts().to_dict())

# Mohler
S = sets['mohler']; S = S[['rid', 'orig_uid', 'group', 'variant', 'keep', 'neg_type', 'student_answer']].copy()
for s in ['ft', 'ftaug', 'instruct_zs']:
    z = pd.read_csv(sc('mohler', s)); e, _, _ = score_from_logprobs(z[LPM].values)
    S = S.merge(pd.DataFrame({'rid': z.rid, s: e}), on='rid')
orig = S[S.variant == 'original'].set_index('orig_uid')
rows = []
for t in ['adds negation word', 'rewrite without negation word']:
    d = S[(S.variant == 'negated_llm') & S.keep & (S.neg_type == t)].set_index('orig_uid')
    if len(d) == 0: continue
    row = {'dataset': 'Mohler', 'type': t, 'n': len(d)}
    for s in ['ft', 'ftaug', 'instruct_zs']:
        row[f'{s} mean change'] = (d[s] - orig.loc[d.index, s]).mean()
    diff = (d.ftaug - orig.loc[d.index, 'ftaug']) - (d.ft - orig.loc[d.index, 'ft'])
    lo, hi, p = boot(diff.values, d.group.values)
    row.update({'ftaug − ft': diff.mean(), 'CI': f'[{lo:.2f}, {hi:.2f}]', 'p': p})
    rows.append(row)

# SciEntsBank (Instruct calibrated exactly as in Notebook 2)
from sklearn.linear_model import LogisticRegression
P = sets['scientsbank'][['rid', 'orig_uid', 'group', 'variant', 'keep', 'neg_type']].copy()
cal = pd.read_csv(f'{RES}/scientsbank_llama3_instruct_zs.csv').drop_duplicates('uid').merge(se[['uid', 'split', 'label']], on='uid')
cal = cal[cal.split.isin(['train', 'val'])]; qc, _, _ = labels_from_logprobs(cal[LPS].values)
zs_cal = LogisticRegression(max_iter=2000).fit(np.log(np.clip(qc, 1e-12, 1)), cal.label)
for s in ['ft', 'ftaug', 'instruct_zs']:
    z = pd.read_csv(sc('scientsbank', s)).set_index('rid').loc[P.rid]; q, _, _ = labels_from_logprobs(z[LPS].values)
    P[s] = zs_cal.predict(np.log(np.clip(q, 1e-12, 1))) if s == 'instruct_zs' else np.array(LABEL_ORDER)[q.argmax(1)]
for t in ['adds negation word', 'rewrite without negation word']:
    d = P[(P.variant == 'negated_llm') & P.keep & (P.neg_type == t)]
    if len(d) == 0: continue
    row = {'dataset': 'SciEntsBank', 'type': t, 'n': len(d)}
    for s in ['ft', 'ftaug', 'instruct_zs']:
        row[f'{s} still correct'] = (d[s] == 'correct').mean()
    diff = (d.ftaug == 'correct').astype(float) - (d.ft == 'correct').astype(float)
    lo, hi, p = boot(diff.values, d.group.values)
    row.update({'ftaug − ft': diff.mean(), 'CI': f'[{lo:.2f}, {hi:.2f}]', 'p': p})
    rows.append(row)
split_tab = pd.DataFrame(rows); split_tab.to_csv(f'{OUT}/T7_heldout_negation_by_type.csv', index=False)
display(split_tab.round(3))

mohler {'rewrite without negation word': 143, 'adds negation word': 56}
scientsbank {'rewrite without negation word': 141, 'adds negation word': 59}


,dataset,type,n,ft mean change,ftaug mean change,instruct_zs mean change,ftaug − ft,CI,p,ft still correct,ftaug still correct,instruct_zs still correct
0,Mohler,adds negation word,56,-3.379,-7.898,-6.267,-4.519,"[-5.43, -3.58]",0.000,NaN,NaN,NaN
1,Mohler,rewrite without negation word,143,-1.937,-3.093,-3.574,-1.156,"[-1.56, -0.73]",0.000,NaN,NaN,NaN
2,SciEntsBank,adds negation word,59,NaN,NaN,NaN,-0.203,"[-0.34, -0.10]",0.000,0.237,0.034,0.220
3,SciEntsBank,rewrite without negation word,141,NaN,NaN,NaN,-0.043,"[-0.09, 0.00]",0.073,0.113,0.071,0.121


Mohler: mean change in score (more negative = better). SciEntsBank: share still labelled `correct` (lower = better). The **rewrite without negation word** rows are the decisive test: they share no surface pattern with the training counter-examples.

## 2. Manual-check sheets

In [3]:
for ds, rs in sets.items():
    path = f'{RES}/manual_check_{ds}.csv'
    if os.path.exists(path):
        continue                                     # never overwrite a sheet you may have filled in
    rng = np.random.default_rng(SEED)
    orig = rs[rs.variant == 'original'].set_index('orig_uid').student_answer
    parts = []
    for v, n in [('negated_llm', 50), ('negated', 20), ('paraphrase', 20)]:
        d = rs[(rs.variant == v) & rs.keep]
        parts.append(d.sample(min(n, len(d)), random_state=SEED))
    sheet = pd.concat(parts)
    sheet = pd.DataFrame({'rid': sheet.rid, 'variant': sheet.variant, 'question': sheet.question,
                          'original': sheet.orig_uid.map(orig).values, 'variant_text': sheet.student_answer,
                          'valid (y/n)': '', 'note': ''})
    sheet.to_csv(path, index=False)
    print('written', path, len(sheet), 'rows')
print('''Fill the "valid (y/n)" column:
  negated_llm / negated: y = the text clearly CONTRADICTS the original answer
  paraphrase:            y = the text keeps EXACTLY the same meaning
Then upload the sheet back to results/ (same name) and re-run this notebook.''')

for ds in sets:
    path = f'{RES}/manual_check_{ds}.csv'
    m = pd.read_csv(path)
    f = m['valid (y/n)'].astype(str).str.strip().str.lower()
    if f.isin(['y', 'n']).any():
        r = m.assign(v=f).query("v in ['y','n']").groupby('variant').v.agg(n='size', valid=lambda x: (x == 'y').mean())
        r.to_csv(f'{OUT}/T8_manual_validity_{ds}.csv'); print(ds); display(r.round(3))

written /content/drive/MyDrive/asag_revision/results/manual_check_mohler.csv 90 rows
written /content/drive/MyDrive/asag_revision/results/manual_check_scientsbank.csv 90 rows
Fill the "valid (y/n)" column:
  negated_llm / negated: y = the text clearly CONTRADICTS the original answer
  paraphrase:            y = the text keeps EXACTLY the same meaning
Then upload the sheet back to results/ (same name) and re-run this notebook.


## 3. Final tables

In [4]:
pm = pd.read_csv(f'{RES}/mohler_preds.csv')
MO = {'mean': 'Mean predictor', 'tfidf_cos': 'TF-IDF cosine', 'minilm_cos': 'MiniLM cosine',
      'composite_fixed': 'Lexical composite, original weights (raw)', 'composite_fixed_cal': 'Lexical composite, original weights (rescaled)',
      'composite_learned': 'Lexical composite, learned weights', 'llama3_base_zs_cal': 'Llama 3 8B zero-shot (rescaled)',
      'llama3_instruct_zs': 'Llama 3 8B Instruct zero-shot', 'llama3_ft': 'Fine-tuned (LoRA)',
      'llama3_ftaug': 'Fine-tuned + counter-examples (LoRA)', 'fusion_original': 'Fusion, original 30/70 rule',
      'fusion_learned': 'Fusion with lexical, learned weights', 'ens_50_50': 'Ensemble fine-tuned + Instruct (50/50)'}
rows = []
for k, name in MO.items():
    r = summarise(pm.y, pm[k], reg_metrics, groups=pm.group, folds=pm.fold)
    corr = k != 'mean'
    rows.append({'System': name, 'Pearson [95% CI]': f"{r['pearson']:.3f} {r['pearson_ci']}" if corr else '–',
                 'Spearman': f"{r['spearman']:.3f}" if corr else '–', 'MAE': f"{r['mae']:.3f}", 'RMSE': f"{r['rmse']:.3f}",
                 'QWK': f"{r['qwk']:.3f}" if corr else '–',
                 'Pearson, folds': f"{r['pearson_fold_mean']:.3f} ± {r['pearson_fold_std']:.3f}" if corr else '–',
                 'Signed error, gold < 6': f"{(pm[k][pm.y < 6] - pm.y[pm.y < 6]).mean():+.2f}"})
h = pd.read_csv(f'{RES}/human_interrater_mohler.csv').iloc[0]
rows.append({'System': 'Human grader 1 vs grader 2 (reference)', 'Pearson [95% CI]': f'{h.pearson:.3f}', 'Spearman': f'{h.spearman:.3f}',
             'MAE': f'{h.mae:.3f}', 'RMSE': f'{h.rmse:.3f}', 'QWK': f'{h.qwk:.3f}', 'Pearson, folds': '–', 'Signed error, gold < 6': '–'})
T1 = pd.DataFrame(rows); T1.to_csv(f'{OUT}/T1_mohler_main.csv', index=False)

sp = pd.read_csv(f'{RES}/scientsbank_preds.csv')
SE = {'majority': 'Majority class', 'lexical_4': 'Lexical features', 'llama3_base_zs_cal': 'Llama 3 8B zero-shot (calibrated)',
      'llama3_instruct_zs': 'Llama 3 8B Instruct zero-shot', 'llama3_instruct_zs_cal': 'Llama 3 8B Instruct zero-shot (calibrated)',
      'llama3_ft': 'Fine-tuned (LoRA)', 'llama3_ftaug': 'Fine-tuned + counter-examples (LoRA)',
      'ens_50_50': 'Ensemble fine-tuned + Instruct (50/50)'}
LC = {'5way': 'label', '3way': 'label3', '2way': 'label2'}; TS = [('test_ua', 'UA'), ('test_uq', 'UQ'), ('test_ud', 'UD')]
wide, long = [], []
for k, name in SE.items():
    row = {'System': name}
    for sch, col in LC.items():
        for t, lab in TS:
            d = sp[sp.split == t].dropna(subset=[f'{k}_{sch}'])
            m = cls_metrics(d[col], d[f'{k}_{sch}'])
            row[f'{sch} {lab} macro-F1'] = f"{m['macro_f1']:.3f}"; row[f'{sch} {lab} weighted-F1'] = f"{m['weighted_f1']:.3f}"
            row[f'{sch} {lab} acc'] = f"{m['acc']:.3f}"
            long.append({'system': name, 'scheme': sch, 'test': lab, **m})
    wide.append(row)
T2 = pd.DataFrame(wide); T2.to_csv(f'{OUT}/T2_scientsbank_main.csv', index=False)
pd.DataFrame(long).to_csv(f'{OUT}/T2_scientsbank_long.csv', index=False)
T2_compact = T2[['System'] + [c for c in T2.columns if 'macro-F1' in c or ('5way' in c and 'weighted' in c)]]
display(T1, T2_compact)

,System,Pearson [95% CI],Spearman,MAE,RMSE,QWK,"Pearson, folds","Signed error, gold < 6"
0,Mean predictor,–,–,1.803,2.215,–,–,+4.43
1,TF-IDF cosine,"0.407 [0.341, 0.473]",0.434,1.543,2.023,0.304,0.418 ± 0.057,+3.87
2,MiniLM cosine,"0.486 [0.425, 0.545]",0.478,1.474,1.935,0.375,0.496 ± 0.093,+3.47
3,"Lexical composite, original weights (raw)","0.542 [0.491, 0.590]",0.571,3.856,4.253,0.185,0.544 ± 0.079,-0.97
4,"Lexical composite, original weights (rescaled)","0.548 [0.491, 0.601]",0.565,1.388,1.856,0.433,0.556 ± 0.086,+3.32
5,"Lexical composite, learned weights","0.543 [0.484, 0.597]",0.558,1.394,1.862,0.428,0.552 ± 0.089,+3.32
6,Llama 3 8B zero-shot (rescaled),"0.611 [0.556, 0.664]",0.608,1.349,1.767,0.497,0.612 ± 0.058,+3.10
7,Llama 3 8B Instruct zero-shot,"0.673 [0.623, 0.725]",0.694,1.625,2.116,0.600,0.674 ± 0.073,+0.30
8,Fine-tuned (LoRA),"0.765 [0.718, 0.809]",0.747,0.969,1.433,0.747,0.793 ± 0.039,+1.83
9,Fine-tuned + counter-examples (LoRA),"0.769 [0.731, 0.804]",0.758,0.966,1.439,0.753,0.776 ± 0.030,+1.47


,System,5way UA macro-F1,5way UA weighted-F1,5way UQ macro-F1,5way UQ weighted-F1,5way UD macro-F1,5way UD weighted-F1,3way UA macro-F1,3way UQ macro-F1,3way UD macro-F1,2way UA macro-F1,2way UQ macro-F1,2way UD macro-F1
0,Majority class,0.121,0.260,0.146,0.239,0.118,0.249,0.210,0.223,0.219,0.362,0.371,0.367
1,Lexical features,0.253,0.431,0.294,0.403,0.265,0.443,0.417,0.450,0.442,0.666,0.685,0.690
2,Llama 3 8B zero-shot (calibrated),0.354,0.516,0.414,0.479,0.366,0.483,0.559,0.541,0.529,0.716,0.693,0.698
3,Llama 3 8B Instruct zero-shot,0.308,0.377,0.326,0.424,0.350,0.422,0.541,0.585,0.573,0.668,0.700,0.687
4,Llama 3 8B Instruct zero-shot (calibrated),0.523,0.587,0.418,0.583,0.572,0.540,0.622,0.652,0.567,0.747,0.806,0.712
5,Fine-tuned (LoRA),0.581,0.738,0.522,0.676,0.720,0.667,0.775,0.746,0.733,0.830,0.820,0.769
6,Fine-tuned + counter-examples (LoRA),0.680,0.764,0.536,0.690,0.702,0.663,0.794,0.771,0.728,0.838,0.845,0.766
7,Ensemble fine-tuned + Instruct (50/50),0.585,0.748,0.524,0.679,0.717,0.661,0.775,0.741,0.733,0.831,0.832,0.765


In [5]:
import shutil
for src, dst in [('TABLE_mohler_lexical_ablation.csv', 'T3_lexical_ablation.csv'), ('mohler_paired_tests.csv', 'T4a_mohler_paired_tests.csv'),
                 ('scientsbank_paired_tests.csv', 'T4b_scientsbank_paired_tests.csv'), ('nb10_ftaug_vs_ft_real_tests.csv', 'T4c_ftaug_vs_ft_real_tests.csv'),
                 ('nb5_fusion_weights.csv', 'T5a_fusion_weights.csv'), ('nb7_mohler_ensemble_weights.csv', 'T5b_ensemble_weights.csv'),
                 ('mohler_error_by_band.csv', 'T5c_mohler_error_by_band.csv'), ('mohler_length_bias.csv', 'T5d_mohler_length_bias.csv')]:
    if os.path.exists(f'{RES}/{src}'): shutil.copy(f'{RES}/{src}', f'{OUT}/{dst}')

# Robustness (T6): both datasets, five systems, plus the paired ftaug − ft difference
rm = pd.read_csv(f'{RES}/robustness2_summary_mohler.csv'); rsb = pd.read_csv(f'{RES}/robustness2_summary_scientsbank.csv')
VAR = ['negated:auxiliary', 'negated:prefix', 'negated_llm', 'keyword_salad', 'swapped', 'typos', 'paraphrase']
SYS = ['ft', 'ftaug', 'instruct_zs', 'ens_50_50', 'lexical']
T6a = rm[rm.system.isin(SYS)].pivot_table(index='variant', columns='system', values='mean_delta').loc[VAR, SYS].round(2)
pa = rm[rm.system == 'ftaug − ft (paired)'].set_index('variant')
T6a['ftaug − ft [CI], p'] = [f"{pa.loc[v, 'mean_delta']:.2f} {pa.loc[v, 'ci']}, p={pa.loc[v, 'p']:.3f}" for v in VAR]
T6b = rsb[rsb.system.isin(SYS)].pivot_table(index='variant', columns='system', values='share_still_correct').loc[['original'] + VAR, SYS].round(3)
pb = rsb[rsb.system == 'ftaug − ft (paired)'].set_index('variant')
T6b['ftaug − ft [CI], p'] = [f"{pb.loc[v, 'share_still_correct']:+.3f} {pb.loc[v, 'ci']}, p={pb.loc[v, 'p']:.3f}" for v in ['original'] + VAR]
T6a.to_csv(f'{OUT}/T6a_robustness_mohler.csv'); T6b.to_csv(f'{OUT}/T6b_robustness_scientsbank.csv')
display(T6a, T6b)

# Cost (T9)
cost = []
for f in ['nb0_latency.csv', 'nb1_llm_latency.csv']:
    if os.path.exists(f'{RES}/{f}'): cost.append(pd.read_csv(f'{RES}/{f}').assign(source=f))
tr = []
for name, files in [('Mohler fine-tune (per fold)', sorted(glob.glob(f'{RES}/nb3_trainlog_f*.json'))),
                    ('Mohler + counter-examples (per fold)', sorted(glob.glob(f'{RES}/nb8_trainlog_f*.json'))),
                    ('SciEntsBank fine-tune', [f'{RES}/nb4_trainlog.json']), ('SciEntsBank + counter-examples', [f'{RES}/nb9_trainlog.json'])]:
    H = [json.load(open(f)) for f in files if os.path.exists(f)]
    if H:
        tr.append({'run': name, 'steps': np.mean([h['total_steps'] for h in H]), 'sec_per_step': np.mean([h['sec_per_step'] for h in H]),
                   'train_minutes': np.mean([h['total_steps'] * h['sec_per_step'] / 60 for h in H]),
                   'peak_vram_gb': np.mean([h['peak_vram_gb'] for h in H]), 'trainable_params': H[0]['trainable_params']})
T9 = pd.DataFrame(tr).round(2); T9.to_csv(f'{OUT}/T9_training_cost.csv', index=False)
if cost: pd.concat(cost).to_csv(f'{OUT}/T9_inference_latency.csv', index=False)
display(T9)

system,ft,ftaug,instruct_zs,ens_50_50,lexical,"ftaug − ft [CI], p"
variant,,,,,,
negated:auxiliary,-1.25,-5.76,-2.64,-1.95,-0.10,"-4.51 [-5.54, -3.41], p=0.001"
negated:prefix,-3.09,-9.27,-6.95,-5.02,-0.09,"-6.19 [-6.69, -5.74], p=0.001"
negated_llm,-2.34,-4.44,-4.33,-3.34,-0.20,"-2.10 [-2.68, -1.58], p=0.001"
keyword_salad,0.21,-7.78,-2.42,-1.10,1.22,"-7.99 [-8.88, -6.94], p=0.001"
swapped,-4.29,-8.12,-7.60,-5.95,-2.35,"-3.84 [-4.27, -3.35], p=0.001"
typos,-0.13,-0.45,-0.77,-0.45,-0.85,"-0.32 [-0.87, 0.02], p=0.103"
paraphrase,-0.20,-0.18,0.09,-0.05,-0.12,"0.02 [-0.05, 0.09], p=0.638"


system,ft,ftaug,instruct_zs,ens_50_50,lexical,"ftaug − ft [CI], p"
variant,,,,,,
original,0.720,0.730,0.760,0.725,0.860,"+0.010 [-0.03, 0.06], p=0.726"
negated:auxiliary,0.381,0.043,0.324,0.367,0.885,"-0.338 [-0.43, -0.25], p=0.001"
negated:prefix,0.148,0.000,0.066,0.115,0.787,"-0.148 [-0.26, -0.06], p=0.001"
negated_llm,0.150,0.060,0.150,0.140,0.785,"-0.090 [-0.14, -0.05], p=0.001"
keyword_salad,0.705,0.005,0.150,0.630,1.000,"-0.700 [-0.81, -0.59], p=0.001"
swapped,0.005,0.000,0.005,0.005,0.055,"-0.005 [-0.02, 0.00], p=0.712"
typos,0.698,0.729,0.656,0.714,0.688,"+0.031 [-0.01, 0.08], p=0.185"
paraphrase,0.696,0.678,0.813,0.731,0.825,"-0.018 [-0.07, 0.04], p=0.612"


,run,steps,sec_per_step,train_minutes,peak_vram_gb,trainable_params
0,Mohler fine-tune (per fold),388.4,4.55,29.44,7.70,41943040
1,Mohler + counter-examples (per fold),466.4,5.15,40.05,7.61,41943040
2,SciEntsBank fine-tune,1118.0,6.09,113.51,8.19,41943040
3,SciEntsBank + counter-examples,1342.0,5.11,114.22,8.18,41943040


In [6]:
# One Markdown file with every table, for drafting the manuscript
parts = ['# Final tables\n']
for f in sorted(glob.glob(f'{OUT}/T*.csv')):
    t = pd.read_csv(f)
    try:
        body = t.to_markdown(index=False, floatfmt='.3f')
    except ImportError:                      # tabulate missing
        body = t.to_csv(index=False)
    parts.append(f'## {os.path.basename(f)[:-4]}\n\n' + body + '\n')
open(f'{OUT}/ALL_TABLES.md', 'w').write('\n'.join(parts))
print('Wrote', len(parts) - 1, 'tables to', OUT)

Wrote 16 tables to /content/drive/MyDrive/asag_revision/results/final_tables
